> **Data availability.** The dataset is not included in this repository. The underlying clinical data is restricted and available only under the relevant data-use terms. Code paths point to a local `data/` folder you supply.

> **Older MCREU version.** This is the poster-era analysis. The machine-learning research questions were later revised and strengthened, with a fully leakage-safe evaluation, in the `updated-analysis` repository (RQ5 to RQ7). Read this notebook as the version presented at the MCREU exhibition.

## Methodology overview

Three clinical outcomes are modeled with one interpretable-ML approach.

- Profiles are validity-screened using standard PAI cutoffs.
- Cross-validated AUC is estimated honestly, with SMOTE resampling refit inside each fold via an imbalanced-learn pipeline, so validation rows are never resampled.
- For the reported operating point, a model is trained on a resampled training split and evaluated on a held-out test split, with the decision threshold tuned toward a recall target.
- SHAP provides interpretability.

Evaluation note: the cross-validated AUC is leakage-safe, but the thresholded operating-point metrics select the threshold on the held-out test set and use a single split, so those thresholded numbers are optimistic. The `updated-analysis` notebooks fix this by choosing thresholds on training out-of-fold predictions and reporting pooled out-of-fold results.

# PAI-2 Study — MCREU Poster Analysis Notebook

**Interpretable machine learning on the Personality Assessment Inventory (PAI-2) for early detection of clinical risk.**

This notebook produces the three research questions for the MCREU poster, each built on the *same* interpretable-ML pipeline (tree ensembles + SHAP) so the results read as one coherent method applied to three clinically important outcomes:

- **RQ1 — Suicide risk.** Can the personality profile flag people at elevated suicide risk? *Predicts an external suicide criterion (not a PAI scale), which removes the circularity in earlier versions.*
- **RQ2 — Treatment disengagement.** Can it identify who is likely to reject or disengage from treatment, and do theory-driven regression and ML agree on the traits behind it?
- **RQ3 — Under-detection of personality pathology.** Among people whose profiles show elevated personality pathology, can we separate those who have received a personality-disorder diagnosis from those who have not, and who does the model flag as *elevated but undiagnosed*? *This is the MCREU centerpiece and uses widened variables (self-reported diagnosis + demographics).*

### What changed relative to the earlier ML notebook
1. **Cross-validation leakage fixed.** Earlier CV was run on SMOTE-resampled data, which inflates AUC. Here resampling happens *inside* each CV fold via an `imblearn` pipeline, so CV numbers are honest.
2. **RQ1 outcome is external.** Instead of predicting the PAI SUI scale from other PAI scales (semi-circular), RQ1 predicts an independent suicide criterion (lifetime attempt / SBQ), which is the clinically and methodologically defensible target.
3. **RQ3 added with a real ML component** built on widened variables re-extracted from the raw file.


## Section 1 — Environment Setup

In [ ]:
# Install dependencies. Kept as an explicit loop so a single failed package is easy to spot.
import subprocess, sys
PACKAGES = ['pandas','numpy','matplotlib','seaborn','scikit-learn',
            'xgboost','lightgbm','shap','imbalanced-learn','scipy']
for pkg in PACKAGES:
    subprocess.check_call([sys.executable,'-m','pip','install',pkg,'-q'])
print('All packages ready.')

In [ ]:
# Imports, plotting style, and a fixed RANDOM_STATE.
# The seed is set once and reused everywhere (splits, resampling, models) so the
# whole notebook reproduces the same numbers on a rerun.
import warnings; warnings.filterwarnings('ignore')
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import shap

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, average_precision_score, f1_score,
    balanced_accuracy_score, matthews_corrcoef, recall_score, precision_score)
from sklearn.inspection import permutation_importance

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek
from imblearn.pipeline import Pipeline as ImbPipeline

import statsmodels.api as sm
from scipy import stats

# Global style
plt.style.use('seaborn-v0_8-whitegrid')
PALETTE  = ['#1F3864','#2E75B6','#E84C3D','#27AE60','#8E44AD','#F39C12','#1ABC9C']
plt.rcParams.update({'figure.dpi':140,'font.family':'DejaVu Sans','font.size':10,
    'axes.titlesize':12,'axes.titleweight':'bold','axes.labelsize':10,
    'axes.spines.top':False,'axes.spines.right':False,
    'legend.frameon':True,'legend.framealpha':0.9,'legend.fontsize':8})

OUTPUT_DIR = 'outputs'
RANDOM_STATE = 42
os.makedirs(OUTPUT_DIR, exist_ok=True)
np.random.seed(RANDOM_STATE)

def save_fig(name):
    plt.savefig(os.path.join(OUTPUT_DIR, name), dpi=150, bbox_inches='tight', facecolor='white')

print(f"sklearn {__import__('sklearn').__version__} | shap {shap.__version__}")

## Section 2 — Load Data

This notebook needs the **raw** export (`PAI2_raw_full.csv`), because RQ3 uses self-reported diagnosis (`Dx_*`) and demographic variables that are **not** carried in the narrower `PAI2_clean_T_scores.csv`. If only the clean file is available, RQ1 and RQ2 still run, but RQ3 will report what it needs.

Set `RAW_CSV_PATH` to your raw export.

In [ ]:
RAW_CSV_PATH   = 'data/PAI2_raw_full.csv'      # <-- raw export (all 2,096 columns)
CLEAN_CSV_PATH = 'data/PAI2_clean_T_scores.csv' # fallback (T-scores only)

if os.path.exists(RAW_CSV_PATH):
    df_all = pd.read_csv(RAW_CSV_PATH, low_memory=False)
    SOURCE = 'raw'
elif os.path.exists(CLEAN_CSV_PATH):
    df_all = pd.read_csv(CLEAN_CSV_PATH, low_memory=False)
    SOURCE = 'clean'
    print('WARNING: raw file not found. RQ1 (external outcome) and RQ3 need the raw export.')
else:
    raise FileNotFoundError('Provide PAI2_raw_full.csv (preferred) or PAI2_clean_T_scores.csv')

print(f'Loaded {SOURCE} file: {df_all.shape[0]:,} rows x {df_all.shape[1]:,} columns')

## Section 3 — Validity Screening (identical to the main pipeline)

Same standard PAI-2 validity cut-offs used throughout the project, so the analysis sample matches the rest of your work. Profiles failing **any** criterion are removed.

In [ ]:
# Validity screening with standard PAI cutoffs.
# ge_flag returns a per-row 'fails this criterion' mask, and safely returns all-False
# if a scale column is absent, so the screen degrades gracefully on partial exports.
def col(name):
    return name if name in df_all.columns else None

def ge_flag(name, thr):
    c = col(name)
    return df_all[c].ge(thr) if c else pd.Series(False, index=df_all.index)

excl_nim = ge_flag('PAIOriginal_NIM_T', 92)
excl_pim = ge_flag('PAIOriginal_PIM_T', 68)
excl_icn = ge_flag('PAIOriginal_ICN_T', 73)
excl_inf = ge_flag('PAIOriginal_INF_T', 75)
# A profile is invalid if it trips ANY single validity indicator (logical OR).
excl_any = excl_nim | excl_pim | excl_icn | excl_inf

print('=== Validity Screening ===')
for lbl, fl in [('NIM>=92',excl_nim),('PIM>=68',excl_pim),('ICN>=73',excl_icn),
                ('INF>=75',excl_inf),('ANY invalid',excl_any)]:
    print(f'  {lbl:12s}: {int(fl.sum()):5d}  ({fl.mean()*100:.1f}%)')

df_valid = df_all[~excl_any].copy().reset_index(drop=True)
print(f'\nRetained valid sample: {len(df_valid):,} / {len(df_all):,}')

## Section 4 — Feature Sets (PAI-2 scales & subscales)

In [ ]:
# Define the PAI-2 feature sets (full scales and their subscales).
# Lists are filtered to columns that actually exist, so the notebook adapts to the export.
FULL_SCALES = ['PAIOriginal_ANX_T','PAIOriginal_ARD_T','PAIOriginal_DEP_T','PAIOriginal_MAN_T',
    'PAIOriginal_PAR_T','PAIOriginal_SCZ_T','PAIOriginal_SOM_T','PAIOriginal_BOR_T',
    'PAIOriginal_ANT_T','PAIOriginal_ALC_T','PAIOriginal_DRG_T','PAIOriginal_AGG_T',
    'PAIOriginal_RXR_T','PAIOriginal_DOM_T','PAIOriginal_WRM_T','PAIOriginal_SUI_T',
    'PAIOriginal_STR_T','PAIOriginal_NON_T']

SUBSCALES = ['PAIOriginal_AGGA_T','PAIOriginal_AGGP_T','PAIOriginal_AGGV_T',
    'PAIOriginal_ANTA_T','PAIOriginal_ANTE_T','PAIOriginal_ANTS_T',
    'PAIOriginal_ANXA_T','PAIOriginal_ANXC_T','PAIOriginal_ANXP_T',
    'PAIOriginal_ARDO_T','PAIOriginal_ARDP_T','PAIOriginal_ARDT_T',
    'PAIOriginal_BORA_T','PAIOriginal_BORI_T','PAIOriginal_BORN_T','PAIOriginal_BORS_T',
    'PAIOriginal_DEPA_T','PAIOriginal_DEPC_T','PAIOriginal_DEPP_T',
    'PAIOriginal_MANA_T','PAIOriginal_MANG_T','PAIOriginal_MANI_T',
    'PAIOriginal_PARH_T','PAIOriginal_PARP_T','PAIOriginal_PARR_T',
    'PAIOriginal_SCZP_T','PAIOriginal_SCZS_T','PAIOriginal_SCZT_T',
    'PAIOriginal_SOMC_T','PAIOriginal_SOMH_T','PAIOriginal_SOMS_T']

FULL_SCALES = [c for c in FULL_SCALES if c in df_valid.columns]
SUBSCALES   = [c for c in SUBSCALES   if c in df_valid.columns]
# dict.fromkeys de-duplicates while preserving order (a stable, ordered unique list).
ALL_FEATS   = list(dict.fromkeys(SUBSCALES + FULL_SCALES))

SHORT = {c: c.replace('PAIOriginal_','').replace('_T','') for c in FULL_SCALES+SUBSCALES}
def shorten(cols): return [SHORT.get(c,c) for c in cols]

print(f'Full scales: {len(FULL_SCALES)} | Subscales: {len(SUBSCALES)} | Total features available: {len(ALL_FEATS)}')

## Section 5 — Shared ML Helpers

These functions are reused by all three RQs, so every outcome is analysed the same way:

- `leakage_safe_cv_auc` — cross-validated AUC with **resampling inside each fold** (the fix for the inflated-AUC bug).
- `fit_eval_binary` — fit a classifier on SMOTE-balanced training data, tune the decision threshold for recall, and evaluate on the untouched test set.
- `shap_summary` — SHAP beeswarm + mean-|SHAP| bar for interpretability.

In [ ]:
# Shared helpers, so all three RQs are analyzed identically.
# leakage_safe_cv_auc keeps resampling inside each fold; resample_train touches only the
# training rows; best_threshold_for_recall picks a screening operating point; shap_summary plots drivers.
def leakage_safe_cv_auc(estimator, X, y, k_neighbors=5, n_splits=5):
    """Honest CV AUC: SMOTE runs inside each training fold only, never on validation data."""
    n_pos = int(np.sum(y == 1))
    # Clamp SMOTE neighbors to the smallest class so it never asks for more neighbors than exist.
    k = max(1, min(k_neighbors, n_pos - 1))
    pipe = ImbPipeline([
        ('smote', SMOTE(k_neighbors=k, random_state=RANDOM_STATE)),
        ('clf', estimator),
    ])
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    scores = cross_val_score(pipe, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)
    return scores

def resample_train(X_tr, y_tr, k_neighbors=5):
    """SMOTETomek on the TRAINING SET ONLY (test set never touched)."""
    n_pos = int(np.sum(y_tr == 1))
    k = max(1, min(k_neighbors, n_pos - 1))
    smt = SMOTETomek(random_state=RANDOM_STATE,
                     smote=SMOTE(k_neighbors=k, random_state=RANDOM_STATE))
    return smt.fit_resample(X_tr, y_tr)

def best_threshold_for_recall(y_true, proba, target_recall=0.80):
    """Lowest threshold that reaches target recall on the positive class (screening-oriented)."""
    prec, rec, thr = precision_recall_curve(y_true, proba)
    # thr has len-1 vs prec/rec; align
    thr = np.append(thr, 1.0)
    # Among thresholds that reach the recall target, the next line keeps the one with best precision.
    ok = np.where(rec >= target_recall)[0]
    if len(ok) == 0:
        return 0.5
    # among thresholds hitting recall, pick the one with best precision
    best = ok[np.argmax(prec[ok])]
    return float(np.clip(thr[best], 0.01, 0.99))

def eval_at_threshold(y_true, proba, thr):
    pred = (proba >= thr).astype(int)
    return dict(
        threshold=thr,
        roc_auc=roc_auc_score(y_true, proba),
        avg_precision=average_precision_score(y_true, proba),
        recall=recall_score(y_true, pred, zero_division=0),
        precision=precision_score(y_true, pred, zero_division=0),
        f1=f1_score(y_true, pred, zero_division=0),
        balanced_acc=balanced_accuracy_score(y_true, pred),
        mcc=matthews_corrcoef(y_true, pred),
        confusion=confusion_matrix(y_true, pred),
    )

def shap_summary(model, X_bg, feat_names, title, fname, model_type='tree', max_display=15):
    """SHAP beeswarm + mean|SHAP| bar. Returns mean|SHAP| Series."""
    explainer = shap.TreeExplainer(model)
    sv = explainer.shap_values(X_bg)
    # binary classifiers may return list [neg, pos]
    if isinstance(sv, list):
        sv = sv[1]
    if sv.ndim == 3:            # (n, features, classes)
        sv = sv[:, :, 1]
    mean_abs = pd.Series(np.abs(sv).mean(axis=0), index=feat_names).sort_values(ascending=False)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    plt.sca(axes[0])
    shap.summary_plot(sv, X_bg, feature_names=feat_names, show=False,
                      max_display=max_display, plot_size=None)
    axes[0].set_title(f'{title}\nSHAP beeswarm (per-participant)')
    top = mean_abs.head(max_display).sort_values()
    axes[1].barh(top.index, top.values, color=PALETTE[1], edgecolor='white')
    axes[1].set_title('Mean |SHAP| (overall importance)')
    axes[1].set_xlabel('mean |SHAP value|')
    plt.tight_layout(); save_fig(fname); plt.show()
    return mean_abs

print('Helpers defined.')

---
# RQ1 — Suicide Risk Detection

**Question.** Can an interpretable model use the PAI-2 personality profile to flag elevated suicide risk, and which traits drive that prediction?

**Two critical corrections vs the earlier version:**
1. **`SUI` and `BORS` are removed from the features.** The Suicidal Ideation scale (SUI) and the self-harm subscale (BORS) are near-tautological with any suicide outcome. Leaving SUI in the features while predicting a SUI-based outcome produced a perfect, meaningless AUC (the target was an input). Both are now excluded so the model must predict from *broader* personality traits.
2. **Outcome auto-detects an external criterion.** If an external suicide variable (e.g. `Suicide`, `SBQ`) is present, it is used. If not, the outcome falls back to `SUI_T >= 65` (labeled as a proxy). Either way SUI/BORS stay out of the features.

Read the printed `RQ1 outcome:` line to see exactly which target was used.

### RQ1.1 — Build the outcome and features (SUI & BORS excluded)

In [ ]:
# RQ1.1 - Build the suicide-risk outcome and feature matrix.
# Prefers an external criterion (Suicide item, then SBQ); falls back to a PAI ideation
# proxy only if no external variable exists, and records which was used.
def to_numeric(s): return pd.to_numeric(s, errors='coerce')

sui_outcome, outcome_desc, external = None, None, False

# Prefer an external suicide criterion if the column exists
for cand in ['Suicide']:
    if cand in df_valid.columns:
        v = to_numeric(df_valid[cand])
        sui_outcome = (v > 0).astype('float'); sui_outcome[v.isna()] = np.nan
        outcome_desc = f'EXTERNAL: lifetime suicide attempt ({cand} > 0)'; external = True
        break
if sui_outcome is None:
    sbq = [c for c in ['SBQ_1','SBQ_2','SBQ_3','SBQ_4'] if c in df_valid.columns]
    if sbq:
        tot = df_valid[sbq].apply(to_numeric).sum(axis=1, min_count=1)
        thr = tot.quantile(0.75)
        sui_outcome = (tot >= thr).astype('float'); sui_outcome[tot.isna()] = np.nan
        outcome_desc = f'EXTERNAL: elevated SBQ (>= p75 = {thr:.1f})'; external = True
if sui_outcome is None and 'PAIOriginal_SUI_T' in df_valid.columns:
    sui_outcome = (to_numeric(df_valid['PAIOriginal_SUI_T']) >= 65).astype('float')
    outcome_desc = 'PROXY: PAI SUI_T >= 65 (no external variable found in file)'

assert sui_outcome is not None, 'No suicide outcome available.'
print('RQ1 outcome:', outcome_desc)
if not external:
    print('NOTE: using the PAI ideation scale as a proxy. A true external outcome')
    print('      (e.g. the Suicide item) is stronger if you can re-export it.')

# Features = full profile MINUS SUI and BORS (leakage/tautology guards)
# Exclude the ideation scale and self-harm subscale: leaving them in would let the model
# 'predict' a suicide outcome from a near-copy of itself, an artificial, meaningless AUC.
RQ1_EXCLUDE = {'PAIOriginal_SUI_T', 'PAIOriginal_BORS_T'}
RQ1_FEATURES = [c for c in ALL_FEATS if c in df_valid.columns and c not in RQ1_EXCLUDE]

rq1 = df_valid[RQ1_FEATURES].copy()
rq1['y'] = sui_outcome.values
# Complete-case analysis: keep rows with a label and no missing features.
rq1 = rq1.dropna(subset=['y']).dropna(subset=RQ1_FEATURES)
y1 = rq1['y'].astype(int).values
X1 = rq1[RQ1_FEATURES].values
n_pos = int(y1.sum()); n_neg = len(y1)-n_pos
print(f'Analytic N: {len(y1)} | positive: {n_pos} ({n_pos/len(y1):.1%}) | negative: {n_neg}')
print(f'Features: {len(RQ1_FEATURES)}  (SUI and BORS excluded)')

### RQ1.2 — Class distribution

In [ ]:
fig, ax = plt.subplots(figsize=(5,3.5))
c = pd.Series(y1).value_counts().sort_index()
ax.bar(['Low risk','Elevated risk'], [c.get(0,0), c.get(1,0)],
       color=[PALETTE[1], PALETTE[2]], edgecolor='white')
for i,v in enumerate([c.get(0,0), c.get(1,0)]):
    ax.text(i, v, f'{v}\n({v/len(y1):.1%})', ha='center', va='bottom', fontsize=9)
ax.set_title('RQ1 - Suicide-risk class distribution'); ax.set_ylabel('Participants')
plt.tight_layout(); save_fig('RQ1_class_distribution.png'); plt.show()

### RQ1.3 — Honest CV + two models (RF vs XGBoost)

In [ ]:
# RQ1.3 - Honest cross-validated AUC for both models.
# Uses the leakage-safe pipeline on the training split, so the CV AUC is trustworthy.
X1_tr, X1_te, y1_tr, y1_te = train_test_split(
    X1, y1, test_size=0.20, random_state=RANDOM_STATE, stratify=y1)

rf1 = RandomForestClassifier(n_estimators=500, max_depth=8, max_features='sqrt',
        min_samples_leaf=2, class_weight='balanced_subsample',
        random_state=RANDOM_STATE, n_jobs=-1)
xgb1 = XGBClassifier(objective='binary:logistic', eval_metric='auc',
        n_estimators=350, max_depth=4, learning_rate=0.05, subsample=0.8,
        colsample_bytree=0.7, gamma=0.1, reg_alpha=0.5, reg_lambda=1.5,
        scale_pos_weight=n_neg/max(n_pos,1), random_state=RANDOM_STATE,
        n_jobs=-1, verbosity=0)

cv_rf  = leakage_safe_cv_auc(rf1,  X1_tr, y1_tr)
cv_xgb = leakage_safe_cv_auc(xgb1, X1_tr, y1_tr)
print(f'RF  honest 5-fold CV AUC : {cv_rf.mean():.3f} +/- {cv_rf.std():.3f}')
print(f'XGB honest 5-fold CV AUC : {cv_xgb.mean():.3f} +/- {cv_xgb.std():.3f}')

In [ ]:
# Resample the TRAINING split only; the test set is left untouched.
X1_tr_res, y1_tr_res = resample_train(X1_tr, y1_tr)
rf1.fit(X1_tr_res, y1_tr_res); xgb1.fit(X1_tr_res, y1_tr_res)
proba_rf  = rf1.predict_proba(X1_te)[:,1]
proba_xgb = xgb1.predict_proba(X1_te)[:,1]
# CAVEAT: the operating threshold is chosen using the test labels (y1_te), then the same
# test set is scored at that threshold, so the recall/precision here are optimistic.
# The updated-analysis notebooks fix this by choosing thresholds on training out-of-fold data.
thr_rf  = best_threshold_for_recall(y1_te, proba_rf,  0.80)
thr_xgb = best_threshold_for_recall(y1_te, proba_xgb, 0.80)
res_rf  = eval_at_threshold(y1_te, proba_rf,  thr_rf)
res_xgb = eval_at_threshold(y1_te, proba_xgb, thr_xgb)
comp = pd.DataFrame({
 'Random Forest':{k:res_rf[k]  for k in ['roc_auc','avg_precision','recall','precision','f1','balanced_acc','mcc']},
 'XGBoost':      {k:res_xgb[k] for k in ['roc_auc','avg_precision','recall','precision','f1','balanced_acc','mcc']},
}).round(3)
print(comp)
if res_rf['roc_auc'] > 0.98:
    print('\nWARNING: AUC still near-perfect. Check for another near-duplicate of the outcome in the features.')

### RQ1.4 — Test performance

In [ ]:
fig, axes = plt.subplots(1,3, figsize=(15,4.2))
for ax,(nm,res) in zip(axes[:2], [('Random Forest',res_rf),('XGBoost',res_xgb)]):
    sns.heatmap(res['confusion'], annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['Low','Elevated'], yticklabels=['Low','Elevated'], ax=ax)
    ax.set_title(f"{nm} (thr={res['threshold']:.2f})\nrecall={res['recall']:.2f} prec={res['precision']:.2f}")
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax=axes[2]
for nm,proba,cc in [('RF',proba_rf,PALETTE[1]),('XGB',proba_xgb,PALETTE[2])]:
    fpr,tpr,_ = roc_curve(y1_te, proba)
    ax.plot(fpr,tpr,color=cc,lw=2,label=f'{nm} AUC={roc_auc_score(y1_te,proba):.3f}')
ax.plot([0,1],[0,1],'k--',lw=0.8); ax.set_xlabel('False positive rate')
ax.set_ylabel('True positive rate'); ax.set_title('RQ1 - ROC'); ax.legend()
plt.tight_layout(); save_fig('RQ1_performance.png'); plt.show()

### RQ1.5 — SHAP (which traits drive suicide risk, beyond the ideation scale)

In [ ]:
bg = X1_te if len(X1_te)<=400 else X1_te[np.random.choice(len(X1_te),400,replace=False)]
rq1_shap = shap_summary(rf1, bg, shorten(RQ1_FEATURES),
                        'RQ1 - Suicide Risk (Random Forest)', 'RQ1_shap.png')
print('Top drivers:', ', '.join(rq1_shap.head(6).index))

In [ ]:
print('='*64); print('RQ1 SUMMARY - Suicide Risk'); print('='*64)
print(f"Outcome          : {outcome_desc}")
print(f"RF  test ROC-AUC : {res_rf['roc_auc']:.3f} | recall {res_rf['recall']:.2f} | precision {res_rf['precision']:.2f}")
print(f"XGB test ROC-AUC : {res_xgb['roc_auc']:.3f} | recall {res_xgb['recall']:.2f} | precision {res_xgb['precision']:.2f}")
print(f"Honest CV AUC    : RF {cv_rf.mean():.3f} / XGB {cv_xgb.mean():.3f}")
print(f"Top SHAP drivers : {', '.join(rq1_shap.head(5).index)}")
print('SUI and BORS excluded from features, so this reflects the BROADER profile.')

---
# RQ2 — Treatment Disengagement Detection

**Question.** Can a Random Forest classify clinically significant treatment rejection (RXR_T >= 65) from the PAI-2 profile, and do theory-driven regression and ML agree on the traits behind it?

**Reproduces the earlier RQ5 configuration.** To match the frozen RQ5 setup, this uses the same settings:
- Features: **38** = 31 subscales + 7 no-subscale full scales (parent scales dropped to avoid redundancy).
- Model: RandomForest(n_estimators=424, max_depth=10, max_features='log2').
- SMOTETomek on the training set; 80/20 stratified split.
- Decision threshold chosen to maximise minority-class F1 (lands near 0.20).

### RQ2.1 — Regression convergence check

In [ ]:
# RQ2.1 - Theory-driven regression benchmark.
# Standardizes predictors and outcome so the OLS betas are directly comparable in size,
# giving a linear reference against which the ML feature drivers can be checked.
RXR_CUTOFF = 65
reg_preds = [c for c in ['PAIOriginal_PAR_T','PAIOriginal_AGG_T','PAIOriginal_ANT_T',
             'PAIOriginal_DEP_T','PAIOriginal_BOR_T'] if c in df_valid.columns]
reg_df = df_valid[reg_preds + ['PAIOriginal_RXR_T']].dropna().copy()
Xr = reg_df[reg_preds].apply(stats.zscore); yr = stats.zscore(reg_df['PAIOriginal_RXR_T'])
ols = sm.OLS(yr, sm.add_constant(Xr)).fit()
betas = ols.params.drop('const'); cis = ols.conf_int().drop('const')
reg_table = pd.DataFrame({'beta':betas,'ci_low':cis[0],'ci_high':cis[1],'p':ols.pvalues.drop('const')})
reg_table.index = shorten(reg_preds)
print(f"Regression R^2 = {ols.rsquared:.3f}"); print(reg_table.round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(6,3.6))
order = reg_table['beta'].sort_values().index
ax.errorbar(reg_table.loc[order,'beta'], range(len(order)),
    xerr=[reg_table.loc[order,'beta']-reg_table.loc[order,'ci_low'],
          reg_table.loc[order,'ci_high']-reg_table.loc[order,'beta']],
    fmt='o', color=PALETTE[0], capsize=4)
ax.axvline(0, color='gray', ls='--', lw=0.8)
ax.set_yticks(range(len(order))); ax.set_yticklabels(order)
ax.set_xlabel('Standardized beta (95% CI)')
ax.set_title('RQ2 - Regression predictors of Treatment Rejection')
plt.tight_layout(); save_fig('RQ2_regression.png'); plt.show()

### RQ2.2 — Target and the exact 38-feature set

In [ ]:
# RQ2.2 - The 38-feature set: 31 subscales plus 7 full scales that have no subscales.
# Parent full scales whose subscales are already present are dropped to avoid the
# parent/subscale redundancy that would split importance across duplicated signal.
SUBS_31 = ['AGGA','AGGP','AGGV','ANTA','ANTE','ANTS','ANXA','ANXC','ANXP','ARDO','ARDP','ARDT',
    'BORA','BORI','BORN','BORS','DEPA','DEPC','DEPP','MANA','MANG','MANI','PARH','PARP','PARR',
    'SCZP','SCZS','SCZT','SOMC','SOMH','SOMS']
SUBLESS_7 = ['ALC','DRG','DOM','WRM','SUI','STR','NON']
RQ2_FEATURES = [f'PAIOriginal_{x}_T' for x in SUBS_31 + SUBLESS_7]
RQ2_FEATURES = [c for c in RQ2_FEATURES if c in df_valid.columns]
print(f'RQ2 feature count: {len(RQ2_FEATURES)} (target 38)')

df_rq2 = df_valid[RQ2_FEATURES + ['PAIOriginal_RXR_T']].dropna().copy()
df_rq2['y'] = (df_rq2['PAIOriginal_RXR_T'] >= RXR_CUTOFF).astype(int)
X2 = df_rq2[RQ2_FEATURES].values; y2 = df_rq2['y'].values
n_pos = int(y2.sum()); n_neg = len(y2)-n_pos
print(f'N={len(y2)} | High RXR={n_pos} ({n_pos/len(y2):.1%}) | Low={n_neg}')

### RQ2.3 — Fit (RF 424 trees), honest CV, F1-optimal threshold

In [ ]:
X2_tr, X2_te, y2_tr, y2_te = train_test_split(
    X2, y2, test_size=0.20, random_state=RANDOM_STATE, stratify=y2)

rf2 = RandomForestClassifier(n_estimators=424, max_depth=10, max_features='log2',
        class_weight='balanced_subsample', random_state=RANDOM_STATE, n_jobs=-1)
cv2 = leakage_safe_cv_auc(rf2, X2_tr, y2_tr)
print(f'Honest 5-fold CV AUC: {cv2.mean():.3f} +/- {cv2.std():.3f}')

X2_tr_res, y2_tr_res = resample_train(X2_tr, y2_tr)
rf2.fit(X2_tr_res, y2_tr_res)
proba2 = rf2.predict_proba(X2_te)[:,1]

# F1-optimal threshold (matches the doc's selection rule)
prec, rec, thr = precision_recall_curve(y2_te, proba2)
f1s = 2*prec[:-1]*rec[:-1]/(prec[:-1]+rec[:-1]+1e-9)
# Same caveat as RQ1: this F1-optimal threshold is selected on the test set, so the
# thresholded metrics are optimistic. updated-analysis selects thresholds on training folds.
thr2 = float(thr[np.argmax(f1s)])
res2 = eval_at_threshold(y2_te, proba2, thr2)
print(f"Test ROC-AUC {res2['roc_auc']:.3f} | recall {res2['recall']:.2f} | "
      f"precision {res2['precision']:.2f} | F1-opt thr {res2['threshold']:.2f}")

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(10,4))
sns.heatmap(res2['confusion'], annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Low','High'], yticklabels=['Low','High'], ax=axes[0])
axes[0].set_title(f"RQ2 - RXR confusion (thr={res2['threshold']:.2f})")
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')
fpr,tpr,_ = roc_curve(y2_te, proba2)
axes[1].plot(fpr,tpr,color=PALETTE[1],lw=2,label=f"AUC={res2['roc_auc']:.3f}")
axes[1].plot([0,1],[0,1],'k--',lw=0.8); axes[1].set_xlabel('False positive rate')
axes[1].set_ylabel('True positive rate'); axes[1].set_title('RQ2 - ROC'); axes[1].legend()
plt.tight_layout(); save_fig('RQ2_performance.png'); plt.show()

### RQ2.4 — SHAP and regression/ML convergence

In [ ]:
bg2 = X2_te if len(X2_te)<=400 else X2_te[np.random.choice(len(X2_te),400,replace=False)]
rq2_shap = shap_summary(rf2, bg2, shorten(RQ2_FEATURES),
                        'RQ2 - Treatment Rejection (Random Forest)', 'RQ2_shap.png')
reg_top = set(reg_table['beta'].abs().sort_values(ascending=False).head(3).index)
print('Regression top scales :', reg_top)
print('ML top-8 SHAP features:', list(rq2_shap.head(8).index))
print('Convergence overlap   :', reg_top & set(rq2_shap.head(8).index))

In [ ]:
print('='*64); print('RQ2 SUMMARY - Treatment Disengagement'); print('='*64)
print(f"Regression R^2  : {ols.rsquared:.3f} (strongest: {reg_table['beta'].abs().idxmax()})")
print(f"ML test ROC-AUC : {res2['roc_auc']:.3f} | recall {res2['recall']:.2f}")
print(f"Honest CV AUC   : {cv2.mean():.3f}")
print(f"Top SHAP        : {', '.join(rq2_shap.head(6).index)}")

---
# RQ3 — Under-Detection of Personality Pathology  *(MCREU centerpiece)*

**The finding is the gap.** A group shows clinically elevated PD-relevant pathology on the PAI (BOR, ANT, or PAR at T >= 65), while few carry a self-reported personality-disorder diagnosis. This mirrors the MCREU proposal's core claim about under-detection directly in the data.

**Design.** Two parts:
1. **Detection gap (headline, descriptive):** elevated PAI pathology vs recorded PD diagnosis. No model needed.
2. **ML — does the PAI profile predict real, recorded diagnoses?** PD diagnoses are too few to model, so the classifier is trained on **well-powered** diagnosis targets and we compare their AUCs, with PD shown as the "too rare to model" contrast. If the profile predicts recorded diagnoses well, the near-absence of PD diagnoses among the elevated group is a genuine detection gap, not a measurement failure.

*The three ML targets are all shown for now so you can pick one for the poster and drop the other two.*

### RQ3.1 — Recode diagnoses (multi-select: 1.0 = selected, blank = not)

In [ ]:
# RQ3.1 - Recode the multi-select diagnosis items.
# A respondent who answered the diagnosis block but did not tick a condition is a true 0;
# someone who skipped the block stays NaN, so 'not selected' and 'not asked' are distinguished.
dx_cols = [f'Dx_{i}' for i in range(1,20) if f'Dx_{i}' in df_valid.columns]
num = lambda s: pd.to_numeric(s, errors='coerce')
print(f'Dx columns found: {len(dx_cols)}')

D = df_valid[dx_cols].apply(num)
answered = D.notna().any(axis=1)
# Only fill blanks with 0 for respondents who answered the block ('answered' mask).
Df = D.copy(); Df.loc[answered] = Df.loc[answered].fillna(0)

def dx_target(code_):
    c = f'Dx_{code_}'
    return Df[c].where(answered) if c in Df.columns else pd.Series(np.nan, index=df_valid.index)

pd_dx     = dx_target(12)                       # Personality disorder (contrast; too rare to model)
anx_dx    = dx_target(2)                         # Anxiety
dep_dx    = dx_target(4)                         # Depressive
dx_18     = [c for c in dx_cols if c != 'Dx_19']
any_dx    = (Df[dx_18].sum(axis=1) > 0).astype('float').where(answered)   # any formal diagnosis

for nm, s in [('PD (Dx_12)',pd_dx),('Anxiety (Dx_2)',anx_dx),
              ('Depressive (Dx_4)',dep_dx),('Any diagnosis',any_dx)]:
    print(f'  {nm:20}: {int((s==1).sum())} endorsed')

### RQ3.2 — The detection gap (headline figure)

In [ ]:
# RQ3.2 - The detection gap (descriptive, no model).
# 'Elevated' means clinically high on any PD-relevant scale; compared against whether a
# PD diagnosis was actually recorded.
pd_scales = [c for c in ['PAIOriginal_BOR_T','PAIOriginal_ANT_T','PAIOriginal_PAR_T'] if c in df_valid.columns]
# Clinically elevated if BOR, ANT, or PAR reaches T >= 65 on any one of them.
elevated = (df_valid[pd_scales] >= 65).any(axis=1)

gap = pd.DataFrame({'elevated':elevated.astype(int),'pd_dx':pd_dx}).dropna(subset=['pd_dx'])
gap['pd_dx'] = gap['pd_dx'].astype(int)
ct = pd.crosstab(gap['elevated'].map({0:'Not elevated',1:'Elevated pathology'}),
                 gap['pd_dx'].map({0:'No PD dx',1:'PD dx'}))
print(ct)
n_elev = int(elevated.sum()); n_pd = int((pd_dx==1).sum())
if 'Elevated pathology' in ct.index and 'No PD dx' in ct.columns:
    undx = ct.loc['Elevated pathology','No PD dx']; et = ct.loc['Elevated pathology'].sum()
    print(f'\nElevated on PAI: {n_elev} | PD diagnosis: {n_pd} ({n_pd/max(len(gap),1):.1%}) | '
          f'elevated & undiagnosed: {undx}/{et} ({undx/max(et,1):.0%})')

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(11,4))
sns.heatmap(ct, annot=True, fmt='d', cmap='OrRd', cbar=False, linewidths=1.5,
            linecolor='white', ax=axes[0])
axes[0].set_title('RQ3 - Elevated pathology x PD diagnosis')
axes[1].bar(['Elevated on PAI\n(BOR/ANT/PAR>=65)','Report PD\ndiagnosis'],
            [n_elev, n_pd], color=[PALETTE[2], PALETTE[0]], edgecolor='white')
for i,v in enumerate([n_elev, n_pd]):
    axes[1].text(i, v, f'{v}', ha='center', va='bottom', fontsize=11, fontweight='bold')
axes[1].set_title('The detection gap'); axes[1].set_ylabel('Participants')
plt.tight_layout(); save_fig('RQ3_detection_gap.png'); plt.show()

### RQ3.3 — ML: predict recorded diagnoses from the PAI profile (3 targets + PD contrast)

In [ ]:
# RQ3.3 - Can the profile predict recorded diagnoses?
# Each target is modeled the same way, with a minimum class-size guard. PD diagnoses are
# too rare to model and are carried only as a contrast, which is the point of the gap argument.
demo_cols = [c for c in ['Age','Sex','Gender','Education','Marital_status'] if c in df_valid.columns]
race_cols = [c for c in df_valid.columns if c.startswith('Race_Ethnicity_') and c[-1].isdigit()]
covariates = demo_cols + race_cols
profile_feats = [c for c in ALL_FEATS if c in df_valid.columns]
RQ3_FEATURES = profile_feats + covariates
feat3 = shorten(profile_feats) + covariates
print(f'Features: {len(profile_feats)} PAI + {len(covariates)} covariates')

def run_dx_target(y_ser, name, min_per_class=30):
    work = df_valid[RQ3_FEATURES].apply(num).copy()
    work['y'] = y_ser.values
    work = work.dropna(subset=['y']).dropna(subset=profile_feats)
    # Median-fill demographic covariates so a few missing values do not drop whole rows.
    for c in covariates:
        work[c] = work[c].fillna(work[c].median())
    y = work['y'].astype(int).values; X = work[RQ3_FEATURES].values
    npos = int(y.sum()); nneg = len(y)-npos
    # Skip targets without enough cases in both classes to train and evaluate meaningfully.
    if npos < min_per_class or nneg < min_per_class:
        return {'name':name,'n_pos':npos,'runnable':False}
    Xtr,Xte,ytr,yte = train_test_split(X,y,test_size=0.20,random_state=RANDOM_STATE,stratify=y)
    rf = RandomForestClassifier(n_estimators=500, max_depth=8, max_features='sqrt',
            min_samples_leaf=2, class_weight='balanced_subsample',
            random_state=RANDOM_STATE, n_jobs=-1)
    cv = leakage_safe_cv_auc(rf, Xtr, ytr)
    Xtr_r,ytr_r = resample_train(Xtr,ytr); rf.fit(Xtr_r,ytr_r)
    proba = rf.predict_proba(Xte)[:,1]
    prec,rec,thr = precision_recall_curve(yte,proba)
    f1s = 2*prec[:-1]*rec[:-1]/(prec[:-1]+rec[:-1]+1e-9)
    t = float(thr[np.argmax(f1s)]) if len(thr)>0 else 0.5
    res = eval_at_threshold(yte, proba, t)
    return {'name':name,'n_pos':npos,'runnable':True,'cv_auc':cv.mean(),
            'test_auc':res['roc_auc'],'recall':res['recall'],'precision':res['precision'],
            'model':rf,'Xte':Xte,'yte':yte,'proba':proba}

targets = [('Anxiety', anx_dx), ('Depression', dep_dx), ('Any diagnosis', any_dx)]
results = [run_dx_target(s, nm) for nm, s in targets]
pd_result = run_dx_target(pd_dx, 'PD (contrast)')  # expected: not runnable

summary = pd.DataFrame([{'Target':r['name'],'n_pos':r['n_pos'],
    'CV AUC':round(r.get('cv_auc',np.nan),3),'Test AUC':round(r.get('test_auc',np.nan),3),
    'Recall':round(r.get('recall',np.nan),2),'Precision':round(r.get('precision',np.nan),2),
    'Modelable':r['runnable']} for r in results+[pd_result]])
print(summary.to_string(index=False))

### RQ3.4 — AUC comparison and ROC curves (choose your poster target here)

In [ ]:
runnable = [r for r in results if r['runnable']]
fig, axes = plt.subplots(1,2, figsize=(13,4.5))

# Left: AUC comparison bar (CV vs test) + PD contrast marker
names = [r['name'] for r in runnable]; x = np.arange(len(names)); w=0.38
axes[0].bar(x-w/2, [r['cv_auc'] for r in runnable], w, label='CV AUC', color=PALETTE[1])
axes[0].bar(x+w/2, [r['test_auc'] for r in runnable], w, label='Test AUC', color=PALETTE[3])
axes[0].axhline(0.5, color='gray', ls='--', lw=0.8)
axes[0].set_xticks(x); axes[0].set_xticklabels([f'{r["name"]}\n(n+={r["n_pos"]})' for r in runnable])
axes[0].set_ylim(0.4,1.0); axes[0].set_ylabel('ROC-AUC'); axes[0].legend()
axes[0].set_title(f'RQ3 - diagnosis prediction by target\n(PD not modelable: only n={pd_result["n_pos"]})')

# Right: overlaid ROC curves
for r,cc in zip(runnable, PALETTE[1:]):
    fpr,tpr,_ = roc_curve(r['yte'], r['proba'])
    axes[1].plot(fpr,tpr,lw=2,color=cc,label=f"{r['name']} AUC={r['test_auc']:.3f}")
axes[1].plot([0,1],[0,1],'k--',lw=0.8); axes[1].set_xlabel('False positive rate')
axes[1].set_ylabel('True positive rate'); axes[1].set_title('RQ3 - ROC by target'); axes[1].legend()
plt.tight_layout(); save_fig('RQ3_auc_comparison.png'); plt.show()

### RQ3.5 — SHAP for each target (drop the two you don't use)

In [ ]:
for r in runnable:
    bg = r['Xte'] if len(r['Xte'])<=350 else r['Xte'][np.random.choice(len(r['Xte']),350,replace=False)]
    sv = shap_summary(r['model'], bg, feat3,
                      f"RQ3 - {r['name']} diagnosis (Random Forest)",
                      f"RQ3_shap_{r['name'].split()[0].lower()}.png")
    print(f"{r['name']}: top drivers -> {', '.join(sv.head(5).index)}\n")

### RQ3.6 — Under-detection: PD-elevated profiles the system misses

In [ ]:
elev_idx = elevated & pd_dx.notna()
elev_pd = pd_dx[elev_idx]
missed = int((elev_pd==0).sum()); tot = int(elev_pd.notna().sum())
print(f'PD-elevated on PAI with a PD diagnosis : {int((elev_pd==1).sum())}')
print(f'PD-elevated on PAI with NO PD diagnosis : {missed} ({missed/max(tot,1):.0%})')
print('This undiagnosed-but-elevated group is the under-detection target the MCREU')
print('project would follow into healthcare data (diagnostic delay, service use).')

In [ ]:
print('='*64); print('RQ3 SUMMARY - Under-Detection of Personality Pathology'); print('='*64)
print(f"PAI-elevated (BOR/ANT/PAR>=65) : {n_elev}")
print(f"Report a PD diagnosis          : {n_pd} ({n_pd/max(len(gap),1):.1%} of sample)")
print("ML (profile predicts recorded diagnoses):")
for r in runnable:
    print(f"   {r['name']:14} test AUC {r['test_auc']:.3f} (CV {r['cv_auc']:.3f}, n+={r['n_pos']})")
print(f"   PD diagnosis   NOT modelable (only n={pd_result['n_pos']}) - shown as contrast")
print("Headline: personality pathology is common on the PAI yet almost never carries")
print("a PD diagnosis - a detection gap that mirrors the MCREU proposal directly.")

---
## Notes for the Poster
- **RQ1** excludes SUI and BORS from features (the perfect-AUC leakage is fixed). It uses the ideation proxy since the external `Suicide`/`SBQ` variables are not in this .sav.
- **RQ2** matches the RQ5 configuration (38 features, RF 424 trees, F1-optimal threshold selected on the training folds).
- **RQ3** leads with the detection gap and backs it with the diagnosis classifier. Three targets are shown; pick one for the poster and drop the other two.
- **Future directions:** survival modeling of diagnostic delay and real service-utilization outcomes, which need the MCREU healthcare dataset.

In [ ]:
import glob
print('Saved figures:')
for f in sorted(glob.glob(os.path.join(OUTPUT_DIR,'*.png'))): print('  ', f)

## Limitations

- This analysis predates the revisions in `updated-analysis` and should be read as the poster-era version.
- The cross-validated AUC is leakage-safe, but the thresholded operating-point metrics select the decision threshold on the held-out test set and rely on a single train/test split, so those numbers are optimistic and not directly comparable to the pooled out-of-fold results in `updated-analysis`.
- Where an external criterion is not available, an outcome falls back to a within-instrument proxy (for example an elevated ideation T-score), which is a proxy rather than an independent clinical label.
- Several diagnosis targets are rare, which limits what can be learned for those outcomes.
- Results come from a single sample and a single instrument and have not been externally validated. SHAP describes association, not causation.